# Reference solution — open only after attempting the lab
Matches `markouts_wrangle_practice.ipynb` and uses only the synthetic CSV.

## 0 · Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('synthetic_trading_events.csv')
raw = pd.read_csv(DATA)
print('Raw:', raw.shape)
display(raw.head(5))

## 1 · Clean

In [ ]:
events = raw.copy()
events['event_type'] = events['event_type'].astype('string').str.strip().str.upper()
events['symbol'] = events['symbol'].astype('string').str.strip().str.upper()
events['side'] = events['side'].astype('string').str.strip().str.upper()
events['recv_ts'] = pd.to_datetime(events['recv_time'], utc=True, errors='coerce', format='mixed')
events['exch_ts'] = pd.to_datetime(events['exch_time'], utc=True, errors='coerce', format='mixed')
events = events.drop_duplicates(subset=['event_id']).sort_values('recv_ts').reset_index(drop=True)
events['session_date'] = events['recv_ts'].dt.strftime('%Y-%m-%d')
print('Clean:', events.shape)
print(events['event_type'].value_counts())
assert events['event_id'].is_unique
assert events['recv_ts'].notna().all()
assert set(events['symbol'].dropna().unique()) == {'BTC-PERP','ETH-PERP'}

## 2 · Split

In [ ]:
btc = events.loc[events['symbol'].eq('BTC-PERP')].copy()
quotes = btc.loc[btc['event_type'].eq('QUOTE'), ['recv_ts','symbol','session_date','mid_price']].copy()
quotes['mid_price'] = pd.to_numeric(quotes['mid_price'], errors='coerce')
quotes = quotes.dropna(subset=['recv_ts','mid_price']).sort_values('recv_ts').reset_index(drop=True)
fills = btc.loc[btc['event_type'].eq('FILL'),['event_id','TTOrderID','cl_ord_id','symbol','session_date','recv_ts','side','price','qty']].copy()
fills['price'] = pd.to_numeric(fills['price'],errors='coerce')
fills['qty'] = pd.to_numeric(fills['qty'],errors='coerce')
fills = fills.rename(columns={'recv_ts':'fill_ts','price':'fill_price'})
fills = fills.dropna(subset=['fill_ts','fill_price','qty','TTOrderID']).copy()
print('Quotes:',len(quotes),'Fills:',len(fills))
display(fills.head(3))

## 3 · Order-event joins

In [ ]:
decisions = btc.loc[btc['event_type'].eq('CANCEL_DECISION'), ['TTOrderID','recv_ts']].rename(columns={'recv_ts':'decision_ts'})
sends = btc.loc[btc['event_type'].eq('CANCEL_SEND'), ['TTOrderID','recv_ts']].rename(columns={'recv_ts':'cancel_send_ts'})
rejects = btc.loc[btc['event_type'].eq('CANCEL_REJECT'), ['TTOrderID','recv_ts']].rename(columns={'recv_ts':'cancel_reject_ts'})
# One event of each category per TTOrderID in this synthetic dataset.
for x in [decisions,sends,rejects]:
    assert x['TTOrderID'].is_unique
fills = fills.merge(decisions,on='TTOrderID',how='left',validate='many_to_one')
fills = fills.merge(sends,on='TTOrderID',how='left',validate='many_to_one')
fills = fills.merge(rejects,on='TTOrderID',how='left',validate='many_to_one')
fills['fill_group'] = np.select(
    [fills['cancel_reject_ts'].isna(), fills['decision_ts'] > fills['fill_ts'],
     fills['cancel_send_ts'] < fills['fill_ts']],
    ['ordinary','decided_too_late','lost_to_latency'], default='review'
)
print(fills['fill_group'].value_counts(dropna=False))
assert fills['fill_group'].ne('review').all()
assert fills['TTOrderID'].is_unique

## 4a · Baseline mid

In [ ]:
# Baseline mid at or immediately before the fill.
left = fills.sort_values('fill_ts').copy()
right = quotes[['recv_ts','symbol','session_date','mid_price']].sort_values('recv_ts').copy()
base = pd.merge_asof(left, right, left_on='fill_ts', right_on='recv_ts',
                     by=['symbol','session_date'], direction='backward',
                     tolerance=pd.Timedelta('250ms'))
base = base.rename(columns={'mid_price':'mid_at_fill','recv_ts':'baseline_quote_ts'})
print('Fill baseline match rate:',base['mid_at_fill'].notna().mean())
assert len(base) == len(fills)

## 4b–5 · Horizon joins and markouts

In [ ]:
enriched = base.copy()
for label, delta in [('100ms','100ms'),('1s','1s'),('10s','10s'),('60s','60s')]:
    enriched['target_ts'] = enriched['fill_ts'] + pd.Timedelta(delta)
    # IMPORTANT: pandas merge_asof needs both inputs globally sorted by time.
    matched = pd.merge_asof(
        enriched.sort_values('target_ts'),
        quotes[['recv_ts','symbol','session_date','mid_price']].sort_values('recv_ts'),
        left_on='target_ts', right_on='recv_ts',
        by=['symbol','session_date'],direction='forward',
        tolerance=pd.Timedelta('250ms')
    )
    enriched = matched.sort_values('event_id').reset_index(drop=True)
    enriched[f'mid_{label}'] = enriched['mid_price']
    enriched = enriched.drop(columns=['recv_ts','mid_price','target_ts'])
enriched['side_sign'] = enriched['side'].map({'BUY':1,'SELL':-1})
for h in ['100ms','1s','10s','60s']:
    enriched[f'markout_bps_{h}'] = (enriched['side_sign']*(enriched[f'mid_{h}']-enriched['fill_price'])/enriched['fill_price']*1e4)
    enriched[f'markout_usd_{h}'] = enriched['side_sign']*(enriched[f'mid_{h}']-enriched['fill_price'])*enriched['qty']
print('Matched 60s:',enriched['mid_60s'].notna().mean())
assert enriched['event_id'].is_unique
assert enriched['mid_60s'].notna().mean() > 0.98
display(enriched[['event_id','fill_group','markout_bps_100ms','markout_bps_1s','markout_bps_10s','markout_bps_60s']].head())

## 5b · PnL summary and plot

In [ ]:
enriched['late'] = enriched['fill_group'] != 'ordinary'
summary = (enriched.groupby('fill_group',as_index=False)
           .agg(n_fills=('event_id','size'),avg_1s_bps=('markout_bps_1s','mean'),
                avg_10s_bps=('markout_bps_10s','mean'),avg_10s_usd=('markout_usd_10s','mean')))
display(summary)
mean_by_late = enriched.groupby('late')['markout_usd_10s'].mean()
extra_cost_per_late_fill = mean_by_late.loc[False] - mean_by_late.loc[True]
late_count = enriched['late'].sum()
num_days = enriched['session_date'].nunique()
est_cost_per_day = extra_cost_per_late_fill*late_count/num_days
print(f'Estimated extra late-fill cost at 10s: ${extra_cost_per_late_fill:.3f}/fill; ${est_cost_per_day:.3f}/day')
ax=summary.plot.bar(x='fill_group',y='avg_10s_bps',legend=False,rot=20,figsize=(8,4))
ax.axhline(0,color='black',linewidth=0.8)
ax.set_ylabel('Mean 10s signed markout (bps)')
plt.tight_layout();plt.show()

## 6 · Day-level uncertainty (optional)

In [ ]:
# Optional uncertainty: resample trading days, not individual fills.
daily = enriched.groupby(['session_date','late']).agg(mean_usd=('markout_usd_10s','mean'),n=('event_id','size')).reset_index()
wide_mean = daily.pivot(index='session_date',columns='late',values='mean_usd')
wide_n = daily.pivot(index='session_date',columns='late',values='n')
daily_cost = (wide_mean[False]-wide_mean[True])*wide_n[True]
daily_cost = daily_cost.dropna()
rng = np.random.default_rng(123)
boot = np.array([rng.choice(daily_cost.to_numpy(),size=len(daily_cost),replace=True).mean() for _ in range(2000)])
print('Day-based estimate (10s):',round(daily_cost.mean(),3),'USD/day')
print('Illustrative 95% bootstrap CI:',np.round(np.quantile(boot,[.025,.975]),3),'USD/day')
print('Note: only',len(daily_cost),'synthetic days, so uncertainty is illustrative.')

## Review your approach
- Did you preserve one row per fill when joining?
- Did you distinguish missing quote mids from expected blanks in non-quote rows?
- Did you check `merge_asof` tolerances and direction?
- Would your cost estimate change if you used 1s instead of 10s?